# wp1pt4pt1g — automatic soft-storey mechanism identification

For each MDOF CBF cyclic-pushover analysis this notebook identifies:

1. the **soft storey(s)** — the storey whose drift runs away while the others plateau
   (*primary* signal), corroborated by **brace fracture** (braces losing axial capacity
   at high displacement), and
2. the **braces to remove** to build the `_ss` (soft-storey) variant — all braces at the
   soft storey(s), as `[bay, level]` pairs (feed to `_remove_braces_and_gussets`).

Algorithm: `phd_project.scripts.mechanism_identification.identify_soft_storey_mechanism`.
Validation data: `E:\03_wp1pt4pt1_dc2_sdof_fitting` (3s, 5s, 7s `*_cbf_dc2_*`, non-`_ss`).
Manual answer to reproduce (3-storey): soft storey **1**, braces **[(1,1), (3,1)]**.

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup & parameters

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from phd_project.scripts.mechanism_identification import identify_soft_storey_mechanism

# --- data location ---
ROOT = Path(r"E:\03_wp1pt4pt1_dc2_sdof_fitting")
EXCLUDE_SUFFIXES = ("_ss", "_appx_sdof", "_sdof")   # keep MDOF folders only

# --- algorithm thresholds (drift primary, brace confirms) ---
DRIFT_CONCENTRATION_FRAC = 0.5  # storey is a candidate if peak drift >= FRAC * max storey drift
FRACTURE_THRESHOLD = 0.30   # brace residual capacity below this = fractured
BRACE_CONFIRM_FRAC = 0.5    # fraction of a storey's braces that must be fractured to confirm
TAIL_FRAC = 0.25            # late-cycle window for the fracture metric

# --- caching (loading the ~300-460 MB pickles is slow; cache results) ---
CACHE_PATH = ROOT / "soft_storey_mechanism_results.json"
FORCE_RECOMPUTE = False

# --- diagnostic-plot palette (color by job, not identity) ---
ACCENT = "#c1121f"    # reserved highlight: soft storey / fractured brace
NEUTRAL = "#9aa0a6"   # everything else
INK = "#222222"

folders = [
    p for p in sorted(ROOT.iterdir())
    if p.is_dir()
    and not p.name.endswith(EXCLUDE_SUFFIXES)
    and (p / "cyclic_pushover" / "recorders.pickle").exists()
]
print(f"{len(folders)} MDOF folders found")
print(", ".join(p.name for p in folders))

## Run the identification (cached)

Each `recorders.pickle` is large, so results are cached to `CACHE_PATH` and saved
incrementally — re-running skips buildings already done (set `FORCE_RECOMPUTE = True`
to redo, e.g. after changing a threshold).

In [ ]:
results = {}
if CACHE_PATH.exists() and not FORCE_RECOMPUTE:
    with open(CACHE_PATH) as f:
        results = json.load(f)

for folder in tqdm(folders, desc="Identifying mechanisms"):
    tag = folder.name
    if tag in results and not FORCE_RECOMPUTE:
        continue
    results[tag] = identify_soft_storey_mechanism(
        folder,
        concentration_frac=DRIFT_CONCENTRATION_FRAC,
        fracture_threshold=FRACTURE_THRESHOLD,
        brace_confirm_frac=BRACE_CONFIRM_FRAC,
        tail_frac=TAIL_FRAC,
    )
    with open(CACHE_PATH, "w") as f:            # incremental save (resumable)
        json.dump(results, f, indent=2)

print(f"{len(results)} results ({CACHE_PATH})")

## Summary table

In [ ]:
rows = []
for tag, r in results.items():
    row = {
        "n_storeys": r["n_storeys"],
        "soft_storeys": tuple(r["soft_storeys"]),
        "braces_to_remove": [tuple(bl) for bl in r["braces_to_remove"]],
    }
    for s, d in r["storey_peak_drift"].items():
        row[f"drift_s{s}"] = d
    rows.append((tag, row))

summary = pd.DataFrame({t: r for t, r in rows}).T
summary.index.name = "tag"
summary = summary.sort_values(["n_storeys", "soft_storeys"])
summary

## Validation vs the manual answer

In [ ]:
# Every 3-storey building was manually identified as first-storey soft with braces
# (1,1) and (3,1). Assert the algorithm reproduces that; report 5s / 7s (unknown a priori).
bad = []
for tag, r in results.items():
    if r["n_storeys"] == 3:
        got_soft = r["soft_storeys"]
        got_braces = sorted(tuple(bl) for bl in r["braces_to_remove"])
        if got_soft != [1] or got_braces != [(1, 1), (3, 1)]:
            bad.append((tag, got_soft, got_braces))

if bad:
    print("MISMATCH for 3-storey buildings:")
    for t, s, b in bad:
        print(f"  {t}: soft={s} braces={b}")
else:
    print("All 3-storey buildings -> soft storey [1], braces [(1,1), (3,1)]  ✓")

print("\n5s / 7s results:")
for tag, r in sorted(results.items()):
    if r["n_storeys"] in (5, 7):
        print(f"  {tag}: soft={r['soft_storeys']} "
              f"confirmed={[s for s, c in r['storey_brace_confirmed'].items() if c]} "
              f"remove={[tuple(bl) for bl in r['braces_to_remove']]}")

## Diagnostic plots

For one representative building per storey count: (left) peak storey drift — the soft
storey concentrates drift; (right) per-brace residual axial capacity — fractured braces
fall below the threshold. The soft storey / fractured braces are drawn in the reserved
accent colour.

In [ ]:
def plot_building(tag, ax_drift, ax_brace):
    r = results[tag]
    soft = set(r["soft_storeys"])

    # --- storey peak drift (horizontal bars, storey on y) ---
    storeys = sorted(int(s) for s in r["storey_peak_drift"])
    drifts = [r["storey_peak_drift"][str(s)] if str(s) in r["storey_peak_drift"]
              else r["storey_peak_drift"][s] for s in storeys]
    colors = [ACCENT if s in soft else NEUTRAL for s in storeys]
    ax_drift.barh(storeys, drifts, color=colors, height=0.65)
    for s, d in zip(storeys, drifts):
        ax_drift.text(d, s, f" {d:.3f}", va="center", ha="left", fontsize=8, color=INK)
    ax_drift.set_yticks(storeys)
    ax_drift.set_ylabel("storey")
    ax_drift.set_xlabel("peak abs. drift [-]")
    ax_drift.set_title(f"{tag}  —  soft storey {sorted(soft)}", fontsize=9)
    ax_drift.margins(x=0.18)

    # --- brace fracture ratio (bars per (bay,level)) ---
    fr = r["brace_fracture_ratio"]
    keys = sorted(fr, key=lambda k: (int(k.split('_')[1]), int(k.split('_')[0])))  # by level, then bay
    vals = [fr[k] for k in keys]
    thr = r["diagnostics"]["fracture_threshold"]
    bcolors = [ACCENT if v < thr else NEUTRAL for v in vals]
    xpos = range(len(keys))
    ax_brace.bar(xpos, vals, color=bcolors, width=0.7)
    ax_brace.axhline(thr, ls="--", lw=1, color=INK)
    ax_brace.text(len(keys) - 0.5, thr, f" thr={thr:g}", va="bottom", ha="right",
                  fontsize=8, color=INK)
    ax_brace.set_xticks(list(xpos))
    ax_brace.set_xticklabels([f"({k.split('_')[0]},{k.split('_')[1]})" for k in keys],
                             rotation=45, ha="right", fontsize=8)
    ax_brace.set_ylabel("residual capacity [-]")
    ax_brace.set_xlabel("brace (bay, level)")
    ax_brace.set_ylim(0, max(1.0, max(vals) * 1.1))
    ax_brace.set_title("brace fracture (accent = fractured)", fontsize=9)


# one representative building per storey count (first of each)
reps = []
for n in (3, 5, 7):
    for tag, r in sorted(results.items()):
        if r["n_storeys"] == n:
            reps.append(tag)
            break

fig, axs = plt.subplots(len(reps), 2, figsize=(10, 3.2 * len(reps)))
axs = np.atleast_2d(axs)
for i, tag in enumerate(reps):
    plot_building(tag, axs[i, 0], axs[i, 1])
for ax in axs.flat:
    ax.grid(axis="x", ls=":", color="0.85", zorder=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
fig.tight_layout()
plt.show()

## Storey-drift heatmap (all buildings)

Peak drift per storey, one row per building (grouped by height). The concentration in a
single storey — the soft storey (outlined) — is visible across the whole set.

In [ ]:
order = sorted(results, key=lambda t: (results[t]["n_storeys"], t))
max_s = max(r["n_storeys"] for r in results.values())
M = np.full((len(order), max_s), np.nan)
soft_cells = []
for i, tag in enumerate(order):
    r = results[tag]
    for s in range(1, r["n_storeys"] + 1):
        M[i, s - 1] = r["storey_peak_drift"].get(str(s), r["storey_peak_drift"].get(s, np.nan))
    for s in r["soft_storeys"]:
        soft_cells.append((i, s - 1))

fig, ax = plt.subplots(figsize=(1.0 + 0.7 * max_s, 0.32 * len(order) + 1))
im = ax.imshow(M, aspect="auto", cmap="magma_r")
ax.set_xticks(range(max_s))
ax.set_xticklabels([f"S{s}" for s in range(1, max_s + 1)])
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=7)
for (i, j) in soft_cells:               # outline the identified soft storey
    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec=ACCENT, lw=2))
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("peak abs. drift [-]")
ax.set_title("Peak storey drift (soft storey outlined)", fontsize=10)
fig.tight_layout()
plt.show()